# Fire Watcher 🛰️🔥 — Treinamento no Google Colab (FLAME 3)

Este notebook executa o pipeline oficial de treinamento e avaliação comparativa dos dois modelos de visão computacional para segmentação binária de fogo (pixel a pixel):
1. **Modelo Óptico (RGB Corrigido):** U-Net com encoder ResNet34 pré-treinado no ImageNet.
2. **Modelo Térmico (Thermal Raw JPG):** U-Net com encoder ResNet34 pré-treinado na paleta térmica.

> **IMPORTANTE:** Certifique-se de ativar a GPU no Colab em: *Ambiente de Execução > Alterar tipo de ambiente de execução > Acelerador de hardware: T4 GPU*.

## 1. Montagem do Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonagem do Repositório e Instalação de Dependências

In [ ]:
import os

# Clona o repositório se ainda não existir no ambiente efêmero
if not os.path.exists('/content/fire-watcher'):
    !git clone https://github.com/mikmathiaz/TCC_FireProject-API.git /content/fire-watcher

%cd /content/fire-watcher
!git pull

# Instalação das dependências isoladas de Machine Learning
!pip install -r ml/requirements-ml.txt -q

## 3. Verificação do Acelerador GPU (CUDA)

In [ ]:
import torch
print(f"Versão PyTorch: {torch.__version__}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ ATENÇÃO: Nenhuma GPU detectada. Ative a GPU em Ambiente de Execução > Alterar tipo.")

## 4. Descompactação do Pacote do Dataset (Bundle .ZIP)

O arquivo `flame3_colab_bundle.zip` deve ser enviado previamente para o seu Google Drive (gerado via `python ml/scripts/prepare_colab_bundle.py`).

In [ ]:
import zipfile
import os

# Ajuste o caminho abaixo conforme o local onde salvou no seu Google Drive
BUNDLE_DRIVE_PATH = "/content/drive/MyDrive/TCC_FireWatcher/flame3_colab_bundle.zip"
DEST_DATA_DIR = "/content/fire-watcher/ml/data/flame3_bundle"

if os.path.exists(BUNDLE_DRIVE_PATH):
    print(f"📦 Descompactando bundle de {BUNDLE_DRIVE_PATH}...")
    os.makedirs(DEST_DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_DRIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall(DEST_DATA_DIR)
    print("✅ Descompactação concluída! Amostras e split prontos.")
else:
    print(f"❌ Arquivo não encontrado: {BUNDLE_DRIVE_PATH}")
    print("Por favor, confirme se o arquivo flame3_colab_bundle.zip está na pasta correta do seu Drive.")

## 5. Modo de Sanidade / Smoke Test (--smoke)

Treina em apenas 8 amostras de fogo por 30 épocas para verificar capacidade de convergência rápida. O IoU deve superar 0.80 para comprovar que o pipeline está 100% íntegro.

In [ ]:
!python ml/scripts/train.py \
    --modality rgb \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --smoke

## 6. Treinamento Modelo 1: RGB Corrigido (40 Épocas)

Os checkpoints serão gravados diretamente no Google Drive para persistência definitiva.

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/TCC_FireWatcher/checkpoints"

!python ml/scripts/train.py \
    --modality rgb \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --output-dir {CHECKPOINT_DIR} \
    --epochs 40 \
    --batch-size 4 \
    --lr 1e-4

## 7. Treinamento Modelo 2: Térmico Raw JPG (40 Épocas)

In [ ]:
!python ml/scripts/train.py \
    --modality thermal \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --output-dir {CHECKPOINT_DIR} \
    --epochs 40 \
    --batch-size 4 \
    --lr 1e-4

## 8. Avaliação Acadêmica no Conjunto de Teste (Test Split)

Avalia ambos os modelos no bloco temporal final (dados futuros nunca vistos pelo modelo) e afere a latência de inferência em ms por imagem.

In [ ]:
print("=== AVALIAÇÃO MODELO RGB ===")
!python ml/scripts/evaluate.py \
    --checkpoint {CHECKPOINT_DIR}/best_model_rgb.pth \
    --modality rgb \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --split test \
    --output-json {CHECKPOINT_DIR}/eval_results_rgb.json

print("\n" + "="*80 + "\n")

print("=== AVALIAÇÃO MODELO TÉRMICO ===")
!python ml/scripts/evaluate.py \
    --checkpoint {CHECKPOINT_DIR}/best_model_thermal.pth \
    --modality thermal \
    --data-dir /content/fire-watcher/ml/data/flame3_bundle \
    --mask-dir /content/fire-watcher/ml/data/flame3_bundle/masks \
    --split-csv /content/fire-watcher/ml/data/flame3_bundle/split.csv \
    --split test \
    --output-json {CHECKPOINT_DIR}/eval_results_thermal.json